# South African multilingual ASR: baseline and fine-tuning (Colab)

Runtime → Change runtime type → **A100 GPU** (L4 works too; halve the batch size in `configs/train_lora.yaml`).

Steps:
1. Put the project on Google Drive and install it
2. Log in to Hugging Face (needed for the gated Swivuriso dataset)
3. Run tests and a smoke run
4. **Baseline**: score existing models on held-out test sets
5. **Fine-tune** Whisper with LoRA
6. Score your fine-tuned model against the baseline

## 1. Project and install

Copy the project folder to Google Drive (e.g. `MyDrive/sa-asr`), then set `PROJECT` below.
Everything the scripts write (eval cache, checkpoints, results) lands inside the project folder,
so it survives Colab disconnects.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/sa-asr'   # <- change if needed
%cd {PROJECT}

In [ ]:
# Model downloads go to fast local disk, not Drive.
import os
os.environ['HF_HOME'] = '/content/hf_cache'
!pip install -q -e ".[dev]"

## 2. Hugging Face login

Swivuriso (`dsfsi-anv/za-african-next-voices`) is gated. First open
https://huggingface.co/datasets/dsfsi-anv/za-african-next-voices, accept the terms,
then log in here with a **read** token. Without access, the scripts skip Swivuriso and warn.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Tests and smoke run (a few minutes)

In [ ]:
!python -m pytest -q
!python scripts/evaluate.py --config configs/eval_smoke.yaml
!python scripts/train.py --config configs/train_smoke.yaml

## 4. Baseline

Scores Whisper large-v3 / turbo (zero-shot), MMS and the dsfsi-anv fine-tune on FLEURS, NCHLT and
Swivuriso test samples (200 utterances per corpus and language). The first run downloads and caches
the eval audio. Rerunning skips anything already scored.

In [ ]:
!python scripts/evaluate.py --config configs/eval_baseline.yaml

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('results/baseline/summary.md', encoding='utf-8').read()))

## 5. Fine-tune

Check `configs/train_lora.yaml` first (languages, corpus weights, steps). A quick trial:
`--max-steps 500 --langs zul xho`. Validation WER per corpus and language is logged every
`eval_steps`, and the best checkpoint by WER is saved to `outputs/.../final`.
If Colab disconnects, rerun the same command with `--resume`.

In [ ]:
!python scripts/train.py --config configs/train_lora.yaml

## 6. Compare against the baseline

Uncomment the `sa-turbo-lora` entry at the bottom of `configs/eval_baseline.yaml`, then rerun.
Only the new model is scored; the existing baseline results are reused.

In [ ]:
!python scripts/evaluate.py --config configs/eval_baseline.yaml --models sa-turbo-lora
display(Markdown(open('results/baseline/summary.md', encoding='utf-8').read()))